<a href="https://colab.research.google.com/github/yueop/AS_LAB/blob/main/FCN_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
from torchvision  import models

class FCN32s(nn.Module):
    def __init__(self, num_classes=21): #PASCAL VOC 클래스 개수 21개 (배경 포합)
        super(FCN32s, self).__init__()

        #1. VGG16 백본 불러오기
        vgg16 = models.vgg16(pretained=True)
        self.features = vgg16.features #5개의 MaxPool을 거치며 1/32 크기로 줄어듦

        #2. FC 계층을 합성곱(Convolution)으로 변환 (논문의 decapitate 및 변환 과정)
        #VGG16의 fc6, fc7을 각각 7x7, 1x1 Conv로 대체
        self.fc6 = nn.Sequential(
            nn.Conv2d(512, 4096, kernel_size=7, padding=3),
            nn.ReLU(inplace=True),
            nn.Dropout2d()
        )
        self.fc7 = nn.Sequential(
            nn.Conv2d(4096, 4096, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Dropout2d()
        )

        #3. 채널 수를 클래스 개수(21)로 맞추는 1x1 합성곱
        self.score_fr = nn.Conv2d(4096, num_classes, kernel_size=1)

        #4. 한 번에 32배로 뻥튀기하는 디컨볼루션
        self.upscore = nn.ConvTranspose2d(num_classes, num_classes,
                                          kernel_size=64, stride=32, padding=16)

        def forward(self, x):
            x = self.features(x)    #(Batch, 512, H/32, W/32)
            x = self.fc6(x)     #(Batch, 4096, H/32, W/32)
            x = self.fc7(x)     #(Batch, 4096, H/32, W/32)

            x = self.score_fr(x)    #(Batch, 21, H/32, W/32) -> 클래스별 점수맵

            out = self.upscore(x)   #(Batch, 21, H, W) -> 32배로 커져서 원본 크기 복원
            return out